In [ ]:
import pandas as pd


In [ ]:
!pip install pandas openpyxl


   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ------ --------------------------------- 1.8/10.8 MB 10.1 MB/s eta 0:00:01
   --------------------------- ------------ 7.3/10.8 MB 19.7 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 18.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------------- -------------------------- 4.2/12.6 MB 19.4 MB/s eta 0:00:01
   ------------------- -------------------- 6.0/12.6 MB 14.2 MB/s eta 0:00:01
   --------------------- ------------------ 6.8/12.6 MB 10.8 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 9.5 MB/s eta 0:00:01
   ------------------------------- -------- 10.0/12.6 MB 9.3 MB/s eta 0:00:01
   -------------------------------------- - 12.1/12.6 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 9.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
df = pd.read_csv("retail_cleaned.csv", header=0)
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'SourceSheet']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.head()


C:\Users\User\AppData\Local\Temp\ipykernel_572\3485512937.py:1: DtypeWarning: Columns (0: invoice) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("retail_cleaned.csv", header=0)


(1033036, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00+05:30,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00+05:30,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00+05:30,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,RECORD FRAME 7 SINGLE SIZE,48,2009-12-01 07:45:00+05:30,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00+05:30,1.65,13085.0,United Kingdom,2010-2011


In [ ]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY
df_clean = df[df['Customer_ID'].notna() & (df['Quantity'] > 0)].copy()
df_clean['InvoiceDate'] = df_clean['InvoiceDate'].dt.tz_localize(None)
snapshot_date = pd.to_datetime("2011-12-10")

rfm = df_clean.groupby('Customer_ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'Quantity': lambda x: (x * df_clean.loc[x.index, 'Price']).sum()
}).reset_index()

rfm.columns = ['Customer_ID', 'Recency', 'Frequency', 'Monetary']
rfm.head()


,Customer_ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [ ]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['F_Score'] >= 3 and row['R_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Lost'
    elif row['R_Score'] >= 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90
rfm.head()


,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,12346.0,325,12,77556.46,2,4,4,At Risk,True
1,12347.0,2,8,4921.53,4,4,4,Champions,False
2,12348.0,75,5,2019.40,3,3,3,Loyal,False
3,12349.0,18,4,4428.69,4,3,4,Loyal,False
4,12350.0,310,1,334.40,2,1,1,Lost,True


In [11]:
# MERGE BACK WITH CLEANED TRANSACTIONS AND EXPORT
df = df[df['Customer_ID'].notna() & (df['Quantity'] > 0)].copy()
df['InvoiceDate'] = df['InvoiceDate'].dt.tz_localize(None)
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID', how='left'
)

df_final.to_csv("Ecommerce_Data_With_RFM.csv", index=False)
print("CSV File saved successfully!")


CSV File saved successfully!
